In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sympy.abc import theta

In [2]:
data = pd.read_parquet("/Users/Euan Bronsky/Downloads/data_final_final.parquet")

df_selected = data.loc[data.groupby(['quote_date', 'moneyness_group', 'maturity_group'])['distance'].idxmin()]

In [3]:
df_selected['constant'] = 1
df_selected['delta_squared'] = df_selected['delta_fun']**2
df_selected['interaction'] = df_selected['delta_fun'] * df_selected['dte']

- **Build the covariance matrix**

In [15]:
# Covariance building function
def build_Q(k, gamma, R):
    p = 5

    # Build Q entry by entry
    Q = np.zeros((p, p))
    for i in range(p):
        for j in range(p):
            Q[i, j] = (
                R[i, j]
                * gamma[i]
                * gamma[j]
                * (1 - np.exp(-(k[i] + k[j])))
                / (k[i] + k[j])
            )

    # Return the result
    return Q

- **Build the correlation matrix**

In [16]:
# Build a valid correlation matrix
def build_R(r_params, p=5):

    # Build a lower-triangular matrix
    L = np.eye(p)
    idx = np.tril_indices(p, k=-1)
    L[idx] = r_params

    # Form a covariance-like matrix
    S = L @ L.T

    # Convert it into a correlation matrix
    d = np.sqrt(np.diag(S))
    R = S / np.outer(d, d)

    # Return the correlation matrix
    return R

- **Kalman estimation function**

In [17]:
# Collapsed Kalman estimation function
def kalman_estimation(par, x):

    # Dimension of the state vector
    p = 5

    # Specify the dates and number of unique dates
    dates = x['quote_date'].unique()
    n_dates = len(dates)

    # Parameters
    k = np.exp(par[:5])
    theta = par[5:10]
    c = theta* (1 - np.exp(-k))
    sigma2 = np.exp(par[10])

    # Covariance matrix
    gamma = np.exp(par[11:16])
    r_params = par[16:26]
    R = build_R(r_params, p=5)
    Q = build_Q(k, gamma, R)

    # Constant transition matrix
    T = np.diag(np.exp(-k))

    # Specify the initial values
    at_init = np.linalg.inv(np.eye(p) - T) @ c
    Pt_init = (np.linalg.inv(np.eye(p**2) - np.kron(T, T)) @ Q.reshape(-1, order = 'F')).reshape((p,p), order = 'F')

    # Create empty arrays
    at_pred = np.zeros((p, n_dates))
    Pt_pred = np.zeros((n_dates, p, p))
    ll = np.zeros(n_dates)

    # Store the initial values
    at_pred[:,0] = at_init
    Pt_pred[0,:,:] = Pt_init

    # Loop over the dates
    for t, date in enumerate(dates):
        group = x[x['quote_date'] == date]

        # Vector of observations and design matrix
        yt = group['iv_fun'].to_numpy(dtype=float)
        Zt = group[['constant', 'delta_fun', 'delta_squared', 'dte', 'interaction']].to_numpy(dtype=float)

        # Covariance matrix of the measurement equation innovation
        Nt = len(yt)
        Ht = sigma2*np.eye(Nt)
        Ht_inv = (1/sigma2)*np.eye(Nt)

        A = Zt.T @ Ht_inv @ Zt
        A_inv = np.linalg.inv(A)

        # Collapsed data vector
        yt_star = A_inv @ Zt.T @ Ht_inv @ yt

        # Remainder term
        et = yt - Zt @ yt_star

        # Prediction error, prediction error variance, and inverse prediction error variance
        Ht_star = A_inv
        vt = yt_star - at_pred[:,t]
        Ft = Pt_pred[t,:,:] + Ht_star
        Ft_inv = np.linalg.inv(Ft)

        # Log-likelihood contribution
        llik_star = -0.5* len(yt_star) * np.log(2*np.pi) - 0.5*np.log(np.linalg.det(Ft)) - 0.5* (vt.T @ Ft_inv @ vt)
        ll[t] = llik_star - 0.5*np.log(np.linalg.det(Ht) / np.linalg.det(Ht_star)) - 0.5* (et.T @ Ht_inv @ et)

        # Compute the Kalman gain, predicted state estimate, and predicted state variance
        if t < n_dates - 1:
            Kt = T @ Pt_pred[t, :, :] @ Ft_inv
            at_pred[:,t+1] = c + T @ at_pred[:,t] + Kt @ vt
            Pt_pred[t+1,:,:] = T @ Pt_pred[t, :, :] @ T.T + Q - Kt @ Ft @ Kt.T

    # Return the results
    return -ll.sum()

- **Minimize the log-likelihood**

In [18]:
# Minimize the log-likelihood function
def minimize_llik(x):

    # Specify initial parameters
    k_ini = [0.2] * 5
    theta_ini = [0.5] * 5
    sigma2_ini = 0.5
    gamma_ini = [0.1] * 5
    rho_ini = [0.0] * 10

    # Create list of initial parameters
    par_ini = list(np.log(k_ini)) +  theta_ini + [np.log(sigma2_ini)] + list(np.log(gamma_ini)) + rho_ini

    counter = {'i': 0}

    def print_iter(xk):
        counter['i'] += 1
        print("Iteration", counter['i'])

    # Minimize the function and obtain parameters and likelihood
    res = minimize(kalman_estimation, par_ini, args=(x,), method = 'BFGS', callback = print_iter)
    par = res.x
    llik = res.fun

    # Return the results
    return par, llik

- **Transform the parameters**

In [19]:
# Transform the parameters
def transform_params(par):

    # Mean-reversion params
    k = np.exp(par[:5])

    # Mean params
    theta = par[5:10]

    # Variance
    sigma_2 = np.exp(par[10])

    # Covariance matrix
    gamma = np.exp(par[11:16])
    r_params = par[16:26]
    R = build_R(r_params, p=5)
    Q = build_Q(k, gamma, R)

    # Return the results
    return k, theta, sigma_2, gamma, Q

- **Obtain the results**

In [ ]:
# Obtain the results
par, llik = minimize_llik(df_selected)

Iteration 1
